In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [3]:
import math
import torch
import torch.nn as nn


class MultiHeadLatentAttention(nn.Module):
    def __init__(
        self,
        d_model: int = 1024,
        n_heads: int = 8,
        d_k: int = 128,
        d_c: int = 32,
        d_hR: int = 32,
    ):
        super().__init__()

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_k
        self.d_c = d_c
        self.d_hR = d_hR

        # Projections for content part
        self.W_DQ = nn.Linear(d_model, d_c)
        self.W_UQ = nn.Linear(d_c, n_heads * d_k)

        self.W_DKV = nn.Linear(d_model, d_c)
        self.W_UK = nn.Linear(d_c, n_heads * d_k)
        self.W_UV = nn.Linear(d_c, n_heads * d_k)

        # Projections for latent / positional part
        self.W_QR = nn.Linear(d_c, n_heads * d_hR)   # -> [B, L, H*d_hR]
        self.W_KR = nn.Linear(d_model, d_hR)         # -> [B, L, d_hR]

        # Output projection
        self.W_O = nn.Linear(n_heads * d_k, d_model)

        # Initialization
        self._reset_parameters()

    def _reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    # Rotary positional embedding on last dimension
    # x: [B, H, L, D] (or [B, 1, L, D])
    def apply_rope(self, x: torch.Tensor) -> torch.Tensor:
        B, H, L, D = x.shape
        device = x.device

        # assume D is even
        half = D // 2
        x1 = x[..., :half]
        x2 = x[..., half:]

        pos = torch.arange(L, dtype=torch.float32, device=device)
        inv_freq = 1.0 / (10000 ** (torch.arange(0, half, 2, device=device).float() / half))

        # [L, half/2]
        freqs = torch.einsum("i,j->ij", pos, inv_freq)
        sin = freqs.sin()
        cos = freqs.cos()

        # [1, 1, L, half]
        sin = torch.stack([sin, sin], dim=-1).reshape(1, 1, L, half)
        cos = torch.stack([cos, cos], dim=-1).reshape(1, 1, L, half)

        # standard 2D rotation on pairs
        x1_even, x1_odd = x1[..., 0::2], x1[..., 1::2]
        x2_even, x2_odd = x2[..., 0::2], x2[..., 1::2]

        # recombine pairs
        x1_rot = torch.empty_like(x1)
        x2_rot = torch.empty_like(x2)

        x1_rot[..., 0::2] = x1_even * cos[..., 0::2] - x1_odd * sin[..., 0::2]
        x1_rot[..., 1::2] = x1_even * sin[..., 0::2] + x1_odd * cos[..., 0::2]

        x2_rot[..., 0::2] = x2_even * cos[..., 0::2] - x2_odd * sin[..., 0::2]
        x2_rot[..., 1::2] = x2_even * sin[..., 0::2] + x2_odd * cos[..., 0::2]

        return torch.cat([x1_rot, x2_rot], dim=-1)

    def forward(self, h_t: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        """
        h_t:  [batch_size, seqlen, d_model]
        mask: [batch_size, seqlen] or broadcastable to [B, 1, 1, L]
              (1 / True = keep, 0 / False = mask)
        """
        batch_size, seqlen, _ = h_t.shape

        # ----- Content queries -----
        c_t_Q = self.W_DQ(h_t)  # [B, L, d_c]
        q_t_c = (
            self.W_UQ(c_t_Q)
            .view(batch_size, seqlen, self.n_heads, self.d_k)
            .transpose(1, 2)
        )  # [B, H, L, d_k]

        # ----- Content keys/values -----
        c_t_KV = self.W_DKV(h_t)  # [B, L, d_c]

        k_t_C = (
            self.W_UK(c_t_KV)
            .view(batch_size, seqlen, self.n_heads, self.d_k)
            .transpose(1, 2)
        )  # [B, H, L, d_k]

        v_t_C = (
            self.W_UV(c_t_KV)
            .view(batch_size, seqlen, self.n_heads, self.d_k)
            .transpose(1, 2)
        )  # [B, H, L, d_k]

        # ----- Latent / RoPE queries -----
        q_t_R = (
            self.W_QR(c_t_Q)
            .view(batch_size, seqlen, self.n_heads, self.d_hR)
            .transpose(1, 2)
        )  # [B, H, L, d_hR]
        q_t_R = self.apply_rope(q_t_R)

        # ----- Latent / RoPE keys -----
        k_t_R = (
            self.W_KR(h_t)
            .view(batch_size, seqlen, 1, self.d_hR)
            .transpose(1, 2)
        )  # [B, 1, L, d_hR]
        k_t_R = self.apply_rope(k_t_R)
        k_t_R = k_t_R.expand(-1, self.n_heads, -1, -1)  # [B, H, L, d_hR]

        # ----- Combine content + latent parts -----
        q_t = torch.cat([q_t_c, q_t_R], dim=-1)  # [B, H, L, d_k + d_hR]
        k_t = torch.cat([k_t_C, k_t_R], dim=-1)  # [B, H, L, d_k + d_hR]

        # ----- Attention scores -----
        attn_scores = torch.matmul(q_t, k_t.transpose(-1, -2))  # [B, H, L, L]
        attn_scores = attn_scores / math.sqrt(self.d_k + self.d_hR)

        # Apply mask if provided
        if mask is not None:
            # make mask broadcastable to [B, H, L, L]
            # assume mask is for keys: [B, L]
            if mask.dim() == 2:
                # [B, 1, 1, L]
                mask_ = mask.unsqueeze(1).unsqueeze(1)
            else:
                mask_ = mask
            attn_scores = attn_scores.masked_fill(mask_ == 0, float("-inf"))

        attn_weights = torch.softmax(attn_scores, dim=-1)  # [B, H, L, L]

        # ----- Apply attention -----
        o_t = torch.matmul(attn_weights, v_t_C)  # [B, H, L, d_k]
        o_t = (
            o_t.transpose(1, 2)
            .contiguous()
            .view(batch_size, seqlen, self.n_heads * self.d_k)
        )  # [B, L, H*d_k]

        u_t = self.W_O(o_t)  # [B, L, d_model]
        return u_t


In [4]:
torch.manual_seed(0)

# Sample "hidden states" from a previous layer
batch_size = 2
seqlen = 5
d_model = 1024

h_t = torch.randn(batch_size, seqlen, d_model)

# Binary padding mask: first sequence is full length,
# second sequence has padding on the last 2 tokens
mask = torch.ones(batch_size, seqlen, dtype=torch.bool)
mask[1, -2:] = 0  # pad last two tokens of second sample

mha = MultiHeadLatentAttention(
    d_model=d_model,
    n_heads=8,
    d_k=128,
    d_c=32,
    d_hR=32,
)

out = mha(h_t, mask=mask)

print("Input h_t shape:", h_t.shape)      # [2, 5, 1024]
print("Output u_t shape:", out.shape)     # [2, 5, 1024]
print("Output sample:", out[0, 0, :5])    # first 5 dims of first token just to see numbers

Input h_t shape: torch.Size([2, 5, 1024])
Output u_t shape: torch.Size([2, 5, 1024])
Output sample: tensor([-0.1298,  0.0430,  0.0303, -0.0098, -0.1725], grad_fn=<SliceBackward0>)
